# Dense scaling laws for addition

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marcoharuni/jax-addition-transformer/blob/main/notebooks/02_dense_scaling_laws.ipynb)

Four dense transformers are trained from scratch at four independent horizons. Every implementation, training loop, checkpoint, table, and figure is in this notebook.


## Setup — pinned Colab T4 runtime

1. Select **Runtime → Change runtime type → T4 GPU**.
2. Run the environment installation cell below. It uses `uv` to create one
   isolated, pinned CUDA 12 environment ahead of Colab's preinstalled packages.
3. Colab restarts the Python process exactly once. After it reconnects, choose
   **Runtime → Run all** (or rerun from this setup cell).
4. Continue only when the validation cell prints `Backend: gpu`.

JAX, JAXlib, the CUDA plugin, and Flax are pinned together. An unbounded
`jax[cuda12]` install can combine JAX 0.11.0 with Flax 0.12.2 and fail at
`from flax import nnx` because JAX 0.11.0 removed `jax.core.Effect`.


In [ ]:
# Generated by scripts/sync_colab_runtime.py; do not hand-edit.
import importlib.metadata as _metadata
import importlib.util as _importlib_util
import json as _json
import os as _os
import platform as _platform
import shutil as _shutil
import signal as _signal
import site as _site
import subprocess as _subprocess
import sys as _sys
from pathlib import Path as _Path

_COLAB_FINGERPRINT = 'jax-0.8.1-flax-0.12.2-cuda12-v1'
_EXPECTED_PYTHON = '3.12'
_UV_BOOTSTRAP_VERSION = '0.11.28'
_REQUIREMENTS = [
    "jax[cuda12]==0.8.1",
    "flax==0.12.2",
    "optax==0.2.6",
    "numpy==2.3.3",
    "pandas==2.3.3",
    "matplotlib==3.10.7",
    "scipy==1.16.3"
]
_EXPECTED_VERSIONS = {
    "flax": "0.12.2",
    "jax": "0.8.1",
    "jax-cuda12-pjrt": "0.8.1",
    "jax-cuda12-plugin": "0.8.1",
    "jaxlib": "0.8.1",
    "matplotlib": "3.10.7",
    "numpy": "2.3.3",
    "optax": "0.2.6",
    "pandas": "2.3.3",
    "scipy": "1.16.3"
}
_IN_COLAB = _importlib_util.find_spec("google.colab") is not None
_ENV_ROOT = _Path("/content/.jax-addition-transformer-colab")
_ENV_PYTHON = _ENV_ROOT / "bin" / "python"
_ENV_SITE_PACKAGES = (
    _ENV_ROOT
    / "lib"
    / f"python{_sys.version_info.major}.{_sys.version_info.minor}"
    / "site-packages"
)
_RESTART_MARKER = _ENV_ROOT / f".{_COLAB_FINGERPRINT}.json"
_PTH_PATH = _Path(_site.getsitepackages()[0]) / "00-jax-addition-colab.pth"
_PTH_CONTENT = (
    "import sys; "
    f"sys.path.insert(0, {str(_ENV_SITE_PACKAGES)!r})\n"
)

if _ENV_SITE_PACKAGES.is_dir():
    _site.addsitedir(str(_ENV_SITE_PACKAGES))
    if str(_ENV_SITE_PACKAGES) in _sys.path:
        _sys.path.remove(str(_ENV_SITE_PACKAGES))
    _sys.path.insert(0, str(_ENV_SITE_PACKAGES))

print("Python:", _platform.python_version())
print("Operating system:", _platform.platform())
print("Google Colab:", _IN_COLAB)
_smi = _subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"],
    check=False,
    capture_output=True,
    text=True,
)
print("Accelerator:", _smi.stdout.strip() or "not detected")

if ".".join(map(str, _sys.version_info[:2])) != _EXPECTED_PYTHON:
    raise RuntimeError(
        f"This notebook requires Python {_EXPECTED_PYTHON}; "
        f"the active runtime is {_platform.python_version()}. "
        "Select Colab's latest runtime and reconnect."
    )
if _IN_COLAB and _smi.returncode != 0:
    raise RuntimeError(
        "No NVIDIA GPU is attached. Select Runtime → Change runtime type → T4 GPU."
    )

def _installed_version(distribution):
    try:
        return _metadata.version(distribution)
    except _metadata.PackageNotFoundError:
        return None

_mismatches = {
    name: (_installed_version(name), expected)
    for name, expected in _EXPECTED_VERSIONS.items()
    if _installed_version(name) != expected
}

if _mismatches:
    if not _IN_COLAB:
        raise RuntimeError(
            "Pinned Colab packages are not active: "
            + _json.dumps(_mismatches, sort_keys=True)
            + "\nCreate an isolated environment from configs/colab-runtime.json."
        )
    if _RESTART_MARKER.exists():
        raise RuntimeError(
            "The pinned installation is still inconsistent after the controlled "
            "restart; refusing to restart again. Details: "
            + _json.dumps(_mismatches, sort_keys=True)
        )

    _uv = _shutil.which("uv")
    if _uv is None:
        print(f"Bootstrapping uv=={_UV_BOOTSTRAP_VERSION}...")
        _subprocess.run(
            [
                _sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                f"uv=={_UV_BOOTSTRAP_VERSION}",
            ],
            check=True,
        )
        _uv = _shutil.which("uv")
    if _uv is None:
        raise RuntimeError("uv was installed but its executable was not found.")

    print("Creating an isolated pinned JAX/Flax environment:", *_REQUIREMENTS)
    _subprocess.run(
        [
            _uv,
            "venv",
            "--clear",
            "--python",
            _sys.executable,
            str(_ENV_ROOT),
        ],
        check=True,
    )
    _subprocess.run(
        [
            _uv,
            "pip",
            "install",
            "--python",
            str(_ENV_PYTHON),
            "--upgrade",
            *_REQUIREMENTS,
        ],
        check=True,
    )
    _subprocess.run(
        [_uv, "pip", "check", "--python", str(_ENV_PYTHON)],
        check=True,
    )

    _metadata_check = (
        "import importlib.metadata as m, json; "
        f"expected={_EXPECTED_VERSIONS!r}; "
        "resolved={name: m.version(name) for name in expected}; "
        "assert resolved == expected, (resolved, expected); "
        "print(json.dumps(resolved, sort_keys=True))"
    )
    _subprocess.run(
        [_ENV_PYTHON, "-c", _metadata_check],
        check=True,
    )
    _PTH_PATH.write_text(_PTH_CONTENT)

    _RESTART_MARKER.write_text(
        _json.dumps(
            {"fingerprint": _COLAB_FINGERPRINT, "packages": _EXPECTED_VERSIONS},
            sort_keys=True,
        )
    )
    print(
        "Pinned packages installed and dependency checks passed. "
        "Colab will now restart once. After reconnecting, run all cells again."
    )
    _os.kill(_os.getpid(), _signal.SIGKILL)

if _IN_COLAB and _ENV_SITE_PACKAGES.is_dir():
    if not _PTH_PATH.is_file() or _PTH_PATH.read_text() != _PTH_CONTENT:
        _PTH_PATH.write_text(_PTH_CONTENT)

print("Pinned Colab environment is already installed; no restart is needed.")


In [ ]:
import gc
import json
import math
import os
import pickle
import subprocess
import time
from pathlib import Path

CACHE_DIR = Path("/content/jax_compilation_cache/scaling_study")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
os.environ["JAX_COMPILATION_CACHE_DIR"] = str(CACHE_DIR)
os.environ["JAX_PERSISTENT_CACHE_MIN_COMPILE_TIME_SECS"] = "0"

# COLAB_RUNTIME_VALIDATE_GPU_V1
_resolved_versions = {
    name: _metadata.version(name)
    for name in (
        "jax",
        "jaxlib",
        "jax-cuda12-plugin",
        "jax-cuda12-pjrt",
        "flax",
        "optax",
        "numpy",
    )
}
assert _resolved_versions == _EXPECTED_VERSIONS, _resolved_versions

# COLAB_RUNTIME_VALIDATE_GPU_V1
_resolved_versions = {
    name: _metadata.version(name)
    for name in _EXPECTED_VERSIONS
}
assert _resolved_versions == _EXPECTED_VERSIONS, _resolved_versions

# COLAB_RUNTIME_VALIDATE_GPU_V1
_resolved_versions = {
    name: _metadata.version(name)
    for name in _EXPECTED_VERSIONS
}
assert _resolved_versions == _EXPECTED_VERSIONS, _resolved_versions

import jax
import jax.numpy as jnp
import numpy as np
import optax
from flax import nnx
import flax as _flax

assert jax.__version__ == _EXPECTED_VERSIONS["jax"]
assert _flax.__version__ == _EXPECTED_VERSIONS["flax"]
assert optax.__version__ == _EXPECTED_VERSIONS["optax"]
assert np.__version__ == _EXPECTED_VERSIONS["numpy"]

_backend = jax.default_backend()
_devices = jax.devices()
if _backend != "gpu" or not any(device.platform == "gpu" for device in _devices):
    raise RuntimeError(
        f"JAX backend is {_backend!r}, not 'gpu'. "
        "Select Runtime → Change runtime type → T4 GPU and run setup again."
    )
_matrix_product = jnp.ones((128, 128), dtype=jnp.float32) @ jnp.ones(
    (128, 128), dtype=jnp.float32
)
jax.block_until_ready(_matrix_product)
if _matrix_product.device.platform != "gpu":
    raise RuntimeError("The validation matrix multiplication did not execute on the GPU.")

_nnx_probe = nnx.Linear(4, 4, rngs=nnx.Rngs(0))
_nnx_graphdef, _nnx_state = nnx.split(_nnx_probe)
_nnx_restored = nnx.merge(_nnx_graphdef, _nnx_state)
assert _nnx_restored(jnp.ones((1, 4), dtype=jnp.float32)).shape == (1, 4)

print("JAX:", _resolved_versions["jax"])
print("JAXlib:", _resolved_versions["jaxlib"])
print("Flax:", _resolved_versions["flax"])
print("Optax:", _resolved_versions["optax"])
print("NumPy:", _resolved_versions["numpy"])
print("Backend:", _backend)
print("Device:", _devices[0].device_kind)
print("GPU matrix multiplication and Flax NNX split/merge: passed")

import jax
import jax.numpy as jnp
import numpy as np
import optax
from flax import nnx

_backend = jax.default_backend()
_devices = jax.devices()
if _backend != "gpu" or not any(device.platform == "gpu" for device in _devices):
    raise RuntimeError(
        f"JAX backend is {_backend!r}, not 'gpu'. "
        "Select Runtime → Change runtime type → T4 GPU and run setup again."
    )
_matrix_product = jnp.ones((128, 128), dtype=jnp.float32) @ jnp.ones(
    (128, 128), dtype=jnp.float32
)
jax.block_until_ready(_matrix_product)
if _matrix_product.device.platform != "gpu":
    raise RuntimeError("The validation matrix multiplication did not execute on the GPU.")

_nnx_probe = nnx.Linear(4, 4, rngs=nnx.Rngs(0))
_nnx_graphdef, _nnx_state = nnx.split(_nnx_probe)
_nnx_restored = nnx.merge(_nnx_graphdef, _nnx_state)
assert _nnx_restored(jnp.ones((1, 4), dtype=jnp.float32)).shape == (1, 4)

print("JAX:", _resolved_versions["jax"])
print("JAXlib:", _resolved_versions["jaxlib"])
print("Flax:", _resolved_versions["flax"])
print("Optax:", _resolved_versions["optax"])
print("NumPy:", _resolved_versions["numpy"])
print("Backend:", _backend)
print("Device:", _devices[0].device_kind)
print("GPU matrix multiplication and Flax NNX split/merge: passed")

# COLAB_RUNTIME_VALIDATE_GPU_V1
_resolved_versions = {
    name: _metadata.version(name)
    for name in _EXPECTED_VERSIONS
}
assert _resolved_versions == _EXPECTED_VERSIONS, _resolved_versions

import jax
import jax.numpy as jnp
import numpy as np
import optax
from flax import nnx
import flax as _flax

assert jax.__version__ == _EXPECTED_VERSIONS["jax"]
assert _flax.__version__ == _EXPECTED_VERSIONS["flax"]
assert optax.__version__ == _EXPECTED_VERSIONS["optax"]
assert np.__version__ == _EXPECTED_VERSIONS["numpy"]

_backend = jax.default_backend()
_devices = jax.devices()
if _backend != "gpu" or not any(device.platform == "gpu" for device in _devices):
    raise RuntimeError(
        f"JAX backend is {_backend!r}, not 'gpu'. "
        "Select Runtime → Change runtime type → T4 GPU and run setup again."
    )
_matrix_product = jnp.ones((128, 128), dtype=jnp.float32) @ jnp.ones(
    (128, 128), dtype=jnp.float32
)
jax.block_until_ready(_matrix_product)
if _matrix_product.device.platform != "gpu":
    raise RuntimeError("The validation matrix multiplication did not execute on the GPU.")

_nnx_probe = nnx.Linear(4, 4, rngs=nnx.Rngs(0))
_nnx_graphdef, _nnx_state = nnx.split(_nnx_probe)
_nnx_restored = nnx.merge(_nnx_graphdef, _nnx_state)
assert _nnx_restored(jnp.ones((1, 4), dtype=jnp.float32)).shape == (1, 4)

print("JAX:", _resolved_versions["jax"])
print("JAXlib:", _resolved_versions["jaxlib"])
print("Flax:", _resolved_versions["flax"])
print("Optax:", _resolved_versions["optax"])
print("NumPy:", _resolved_versions["numpy"])
print("Backend:", _backend)
print("Device:", _devices[0].device_kind)
print("GPU matrix multiplication and Flax NNX split/merge: passed")

import jax
import jax.numpy as jnp
import numpy as np
import optax
from flax import nnx

_backend = jax.default_backend()
_devices = jax.devices()
if _backend != "gpu" or not any(device.platform == "gpu" for device in _devices):
    raise RuntimeError(
        f"JAX backend is {_backend!r}, not 'gpu'. "
        "Select Runtime → Change runtime type → T4 GPU and run setup again."
    )
_matrix_product = jnp.ones((128, 128), dtype=jnp.float32) @ jnp.ones(
    (128, 128), dtype=jnp.float32
)
jax.block_until_ready(_matrix_product)
if _matrix_product.device.platform != "gpu":
    raise RuntimeError("The validation matrix multiplication did not execute on the GPU.")

_nnx_probe = nnx.Linear(4, 4, rngs=nnx.Rngs(0))
_nnx_graphdef, _nnx_state = nnx.split(_nnx_probe)
_nnx_restored = nnx.merge(_nnx_graphdef, _nnx_state)
assert _nnx_restored(jnp.ones((1, 4), dtype=jnp.float32)).shape == (1, 4)

print("JAX:", _resolved_versions["jax"])
print("JAXlib:", _resolved_versions["jaxlib"])
print("Flax:", _resolved_versions["flax"])
print("Optax:", _resolved_versions["optax"])
print("NumPy:", _resolved_versions["numpy"])
print("Backend:", _backend)
print("Device:", _devices[0].device_kind)
print("GPU matrix multiplication and Flax NNX split/merge: passed")

# COLAB_RUNTIME_VALIDATE_GPU_V1
_resolved_versions = {
    name: _metadata.version(name)
    for name in _EXPECTED_VERSIONS
}
assert _resolved_versions == _EXPECTED_VERSIONS, _resolved_versions

# COLAB_RUNTIME_VALIDATE_GPU_V1
_resolved_versions = {
    name: _metadata.version(name)
    for name in _EXPECTED_VERSIONS
}
assert _resolved_versions == _EXPECTED_VERSIONS, _resolved_versions

import jax
import jax.numpy as jnp
import numpy as np
import optax
from flax import nnx
import flax as _flax

assert jax.__version__ == _EXPECTED_VERSIONS["jax"]
assert _flax.__version__ == _EXPECTED_VERSIONS["flax"]
assert optax.__version__ == _EXPECTED_VERSIONS["optax"]
assert np.__version__ == _EXPECTED_VERSIONS["numpy"]

_backend = jax.default_backend()
_devices = jax.devices()
if _backend != "gpu" or not any(device.platform == "gpu" for device in _devices):
    raise RuntimeError(
        f"JAX backend is {_backend!r}, not 'gpu'. "
        "Select Runtime → Change runtime type → T4 GPU and run setup again."
    )
_matrix_product = jnp.ones((128, 128), dtype=jnp.float32) @ jnp.ones(
    (128, 128), dtype=jnp.float32
)
jax.block_until_ready(_matrix_product)
if _matrix_product.device.platform != "gpu":
    raise RuntimeError("The validation matrix multiplication did not execute on the GPU.")

_nnx_probe = nnx.Linear(4, 4, rngs=nnx.Rngs(0))
_nnx_graphdef, _nnx_state = nnx.split(_nnx_probe)
_nnx_restored = nnx.merge(_nnx_graphdef, _nnx_state)
assert _nnx_restored(jnp.ones((1, 4), dtype=jnp.float32)).shape == (1, 4)

print("JAX:", _resolved_versions["jax"])
print("JAXlib:", _resolved_versions["jaxlib"])
print("Flax:", _resolved_versions["flax"])
print("Optax:", _resolved_versions["optax"])
print("NumPy:", _resolved_versions["numpy"])
print("Backend:", _backend)
print("Device:", _devices[0].device_kind)
print("GPU matrix multiplication and Flax NNX split/merge: passed")

import jax
import jax.numpy as jnp
import numpy as np
import optax
from flax import nnx

_backend = jax.default_backend()
_devices = jax.devices()
if _backend != "gpu" or not any(device.platform == "gpu" for device in _devices):
    raise RuntimeError(
        f"JAX backend is {_backend!r}, not 'gpu'. "
        "Select Runtime → Change runtime type → T4 GPU and run setup again."
    )
_matrix_product = jnp.ones((128, 128), dtype=jnp.float32) @ jnp.ones(
    (128, 128), dtype=jnp.float32
)
jax.block_until_ready(_matrix_product)
if _matrix_product.device.platform != "gpu":
    raise RuntimeError("The validation matrix multiplication did not execute on the GPU.")

_nnx_probe = nnx.Linear(4, 4, rngs=nnx.Rngs(0))
_nnx_graphdef, _nnx_state = nnx.split(_nnx_probe)
_nnx_restored = nnx.merge(_nnx_graphdef, _nnx_state)
assert _nnx_restored(jnp.ones((1, 4), dtype=jnp.float32)).shape == (1, 4)

print("JAX:", _resolved_versions["jax"])
print("JAXlib:", _resolved_versions["jaxlib"])
print("Flax:", _resolved_versions["flax"])
print("Optax:", _resolved_versions["optax"])
print("NumPy:", _resolved_versions["numpy"])
print("Backend:", _backend)
print("Device:", _devices[0].device_kind)
print("GPU matrix multiplication and Flax NNX split/merge: passed")

# COLAB_RUNTIME_VALIDATE_GPU_V1
_resolved_versions = {
    name: _metadata.version(name)
    for name in _EXPECTED_VERSIONS
}
assert _resolved_versions == _EXPECTED_VERSIONS, _resolved_versions

import jax
import jax.numpy as jnp
import numpy as np
import optax
from flax import nnx
import flax as _flax

assert jax.__version__ == _EXPECTED_VERSIONS["jax"]
assert _flax.__version__ == _EXPECTED_VERSIONS["flax"]
assert optax.__version__ == _EXPECTED_VERSIONS["optax"]
assert np.__version__ == _EXPECTED_VERSIONS["numpy"]

_backend = jax.default_backend()
_devices = jax.devices()
if _backend != "gpu" or not any(device.platform == "gpu" for device in _devices):
    raise RuntimeError(
        f"JAX backend is {_backend!r}, not 'gpu'. "
        "Select Runtime → Change runtime type → T4 GPU and run setup again."
    )
_matrix_product = jnp.ones((128, 128), dtype=jnp.float32) @ jnp.ones(
    (128, 128), dtype=jnp.float32
)
jax.block_until_ready(_matrix_product)
if _matrix_product.device.platform != "gpu":
    raise RuntimeError("The validation matrix multiplication did not execute on the GPU.")

_nnx_probe = nnx.Linear(4, 4, rngs=nnx.Rngs(0))
_nnx_graphdef, _nnx_state = nnx.split(_nnx_probe)
_nnx_restored = nnx.merge(_nnx_graphdef, _nnx_state)
assert _nnx_restored(jnp.ones((1, 4), dtype=jnp.float32)).shape == (1, 4)

print("JAX:", _resolved_versions["jax"])
print("JAXlib:", _resolved_versions["jaxlib"])
print("Flax:", _resolved_versions["flax"])
print("Optax:", _resolved_versions["optax"])
print("NumPy:", _resolved_versions["numpy"])
print("Backend:", _backend)
print("Device:", _devices[0].device_kind)
print("GPU matrix multiplication and Flax NNX split/merge: passed")

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax
import pandas as pd
from flax import nnx
from google.colab import drive
from IPython.display import display

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/jax-addition-transformer")
STUDY_ROOT = DRIVE_ROOT / "scaling"
FIGURE_ROOT = STUDY_ROOT / "figures"
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.2,
})


def atomic_write_bytes(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_bytes(payload)
    os.replace(temporary, path)

def atomic_write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2) + "\n")
    os.replace(temporary, path)

def save_figure(figure, name):
    path = FIGURE_ROOT / f"{name}.png"
    figure.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    print("Saved:", path)


## Vocabulary and experiment grid

Each record is `aaa + bbb = dddd`, with the four answer digits reversed so generation follows the carry direction. Only those four answer positions contribute to the loss.


In [ ]:
SEED = 42
VOCAB_SIZE = 13
MAX_SEQUENCE_LENGTH = 16
MODEL_INPUT_LENGTH = 15
PROMPT_LENGTH = 12
ANSWER_DIGITS = 4

TRAIN_SIZE = 200_000
VALIDATION_SIZE = 20_000
TEST_SIZE = 780_000
BATCH_SIZE = 2048
EVAL_BATCH_SIZE = 4000

HORIZONS = (50, 125, 300, 750)
PEAK_LR = 1e-3
FINAL_LR = 1e-4
WEIGHT_DECAY = 0.1
GRAD_CLIP = 1.0
PARAM_DTYPE = jnp.float32
COMPUTE_DTYPE = jnp.float16

TOKENS = "0123456789 +="
TOKEN_TO_ID = {token: index for index, token in enumerate(TOKENS)}
ID_TO_TOKEN = {index: token for token, index in TOKEN_TO_ID.items()}

def encode(text):
    assert all(character in TOKEN_TO_ID for character in text)
    return np.asarray([TOKEN_TO_ID[character] for character in text], dtype=np.int32)

def decode(token_ids):
    token_ids = np.asarray(token_ids)
    assert token_ids.ndim == 1
    assert np.all((0 <= token_ids) & (token_ids < VOCAB_SIZE))
    return "".join(ID_TO_TOKEN[int(token_id)] for token_id in token_ids)

def format_prompt(a, b):
    assert 0 <= a <= 999 and 0 <= b <= 999
    return f"{a:03d} + {b:03d} = "

def format_example(a, b):
    answer = f"{a + b:04d}"
    return format_prompt(a, b) + answer[::-1]

for a, b in [(0, 0), (7, 42), (99, 1), (123, 456), (999, 999)]:
    example = format_example(a, b)
    assert len(example) == MAX_SEQUENCE_LENGTH
    print(example)

assert len(TOKENS) == VOCAB_SIZE
assert TRAIN_SIZE + VALIDATION_SIZE + TEST_SIZE == 1_000_000


In [ ]:
DENSE_CONFIGS = {
    "dense_0p16m": {"n_layers": 2, "d_model": 64, "n_heads": 1, "d_ff": 496,
                     "parameter_count": 162_176},
    "dense_0p64m": {"n_layers": 2, "d_model": 128, "n_heads": 2, "d_ff": 992,
                     "parameter_count": 643_840},
    "dense_2p16m": {"n_layers": 3, "d_model": 192, "n_heads": 3, "d_ff": 1488,
                     "parameter_count": 2_164_608},
    "dense_10m": {"n_layers": 5, "d_model": 320, "n_heads": 5, "d_ff": 2480,
                   "parameter_count": 10_000_000},
}

study = {
    "model_sizes_per_architecture": len(DENSE_CONFIGS),
    "independent_horizons": list(HORIZONS),
    "runs_per_architecture": len(DENSE_CONFIGS) * len(HORIZONS),
}
assert study["model_sizes_per_architecture"] == 4
assert study["independent_horizons"] == [50, 125, 300, 750]
assert study["runs_per_architecture"] == 16

configuration_table = pd.DataFrame([
    {"model": name, "layers": config["n_layers"], "width": config["d_model"],
     "heads": config["n_heads"], "FFN width": config["d_ff"],
     "parameters": config["parameter_count"]}
    for name, config in DENSE_CONFIGS.items()
])
exposure_table = pd.DataFrame({"horizon": HORIZONS})
exposure_table["examples"] = exposure_table["horizon"] * BATCH_SIZE
exposure_table["input tokens"] = exposure_table["examples"] * MODEL_INPUT_LENGTH
exposure_table["answer tokens"] = exposure_table["examples"] * ANSWER_DIGITS
display(configuration_table.style.format({"parameters": "{:,}"}))
display(exposure_table.style.format({"examples": "{:,}", "input tokens": "{:,}",
                                     "answer tokens": "{:,}"}))


## Addition dataset


In [ ]:
def pair_ids_to_operands(pair_ids):
    pair_ids = np.asarray(pair_ids, dtype=np.int32)
    return pair_ids // 1000, pair_ids % 1000

def operand_lengths(values):
    values = np.asarray(values)
    return np.where(values < 10, 1, np.where(values < 100, 2, 3)).astype(np.int8)

def carry_codes(a, b):
    a = np.asarray(a)
    b = np.asarray(b)
    units = ((a % 10) + (b % 10) >= 10).astype(np.int8)
    tens = (((a // 10) % 10) + ((b // 10) % 10) + units >= 10).astype(np.int8)
    hundreds = (((a // 100) % 10) + ((b // 100) % 10) + tens >= 10).astype(np.int8)
    return (4 * units + 2 * tens + hundreds).astype(np.int8)

def stratum_codes(a, b):
    return (
        ((operand_lengths(a) - 1) * 3 + (operand_lengths(b) - 1)) * 8
        + carry_codes(a, b)
    ).astype(np.int16)

def make_sequences(pair_ids):
    pair_ids = np.asarray(pair_ids, dtype=np.int32)
    a, b = pair_ids_to_operands(pair_ids)
    total = a + b
    sequences = np.empty((len(pair_ids), MAX_SEQUENCE_LENGTH), dtype=np.uint8)
    sequences[:, 0] = a // 100
    sequences[:, 1] = (a // 10) % 10
    sequences[:, 2] = a % 10
    sequences[:, 3:6] = (10, 11, 10)
    sequences[:, 6] = b // 100
    sequences[:, 7] = (b // 10) % 10
    sequences[:, 8] = b % 10
    sequences[:, 9:12] = (10, 12, 10)
    sequences[:, 12] = total % 10
    sequences[:, 13] = (total // 10) % 10
    sequences[:, 14] = (total // 100) % 10
    sequences[:, 15] = (total // 1000) % 10
    return sequences

def largest_remainder(counts, total, capacity):
    ideal = counts.astype(np.float64) * total / counts.sum()
    allocated = np.minimum(np.floor(ideal).astype(np.int64), capacity)
    order = np.lexsort((np.arange(len(counts)), -(ideal - allocated)))
    remaining = total - int(allocated.sum())
    while remaining:
        eligible = order[allocated[order] < capacity[order]]
        take = eligible[:remaining]
        allocated[take] += 1
        remaining -= len(take)
    return allocated

def build_split(seed=SEED):
    pair_ids = np.arange(1_000_000, dtype=np.int32)
    a, b = pair_ids_to_operands(pair_ids)
    codes = stratum_codes(a, b)
    unique_codes, counts = np.unique(codes, return_counts=True)
    train_counts = largest_remainder(counts, TRAIN_SIZE, counts)
    validation_counts = largest_remainder(counts, VALIDATION_SIZE, counts - train_counts)
    rng = np.random.default_rng(seed)
    train_parts, validation_parts, test_parts = [], [], []
    for code_value, train_count, validation_count in zip(
        unique_codes, train_counts, validation_counts, strict=True
    ):
        members = pair_ids[codes == code_value].copy()
        rng.shuffle(members)
        train_parts.append(members[:train_count])
        validation_parts.append(members[train_count:train_count + validation_count])
        test_parts.append(members[train_count + validation_count:])
    train_ids = np.concatenate(train_parts)
    validation_ids = np.concatenate(validation_parts)
    test_ids = np.concatenate(test_parts)
    rng.shuffle(train_ids)
    rng.shuffle(validation_ids)
    rng.shuffle(test_ids)
    assert (len(train_ids), len(validation_ids), len(test_ids)) == (
        TRAIN_SIZE, VALIDATION_SIZE, TEST_SIZE
    )
    assert len(np.unique(np.concatenate([train_ids, validation_ids, test_ids]))) == 1_000_000
    return train_ids, validation_ids, test_ids

train_ids, validation_ids, test_ids = build_split()
train_sequences = make_sequences(train_ids)
train_a, train_b = pair_ids_to_operands(train_ids)
train_strata = stratum_codes(train_a, train_b)

print(f"Train: {len(train_ids):,}")
print(f"Validation: {len(validation_ids):,}")
print(f"Test: {len(test_ids):,}")
print(f"Cached training data: {train_sequences.nbytes / 1e6:.1f} MB")


In [ ]:
class HybridBatcher:
    def __init__(self, sequences, strata, batch_size, seed):
        self.sequences = sequences
        self.batch_size = batch_size
        self.rng = np.random.default_rng(seed)
        self.stratum_indices = [
            np.flatnonzero(strata == code) for code in np.unique(strata)
        ]

    def sample(self):
        natural_count = self.batch_size // 2
        balanced_count = self.batch_size - natural_count
        natural = self.rng.integers(0, len(self.sequences), size=natural_count)
        chosen_strata = self.rng.integers(0, len(self.stratum_indices), size=balanced_count)
        balanced = np.empty(balanced_count, dtype=np.int64)
        for stratum in np.unique(chosen_strata):
            locations = np.flatnonzero(chosen_strata == stratum)
            balanced[locations] = self.rng.choice(
                self.stratum_indices[int(stratum)], size=len(locations), replace=True
            )
        indices = np.concatenate([natural, balanced])
        self.rng.shuffle(indices)
        batch = self.sequences[indices].astype(np.int32)
        return batch[:, :-1], batch[:, 1:]

sample_batcher = HybridBatcher(train_sequences, train_strata, BATCH_SIZE, SEED)
sample_inputs, sample_targets = sample_batcher.sample()
assert sample_inputs.shape == sample_targets.shape == (BATCH_SIZE, MODEL_INPUT_LENGTH)

figure, axes = plt.subplots(1, 2, figsize=(12, 3.5))
example = format_example(123, 456)
for position, token in enumerate(example):
    axes[0].add_patch(plt.Rectangle((position, 0), 0.9, 0.8, fill=False))
    axes[0].text(position + 0.45, 0.4, "space" if token == " " else token,
                 ha="center", va="center", fontsize=8)
axes[0].set(xlim=(0, 16), ylim=(0, 1), title="123 + 456 = 9750")
axes[0].axis("off")

sample_ids = train_ids[np.random.default_rng(SEED).integers(0, len(train_ids), 50_000)]
sample_a, sample_b = pair_ids_to_operands(sample_ids)
natural = np.bincount(carry_codes(sample_a, sample_b), minlength=8) / 50_000
batch_a = sample_inputs[:, 0] * 100 + sample_inputs[:, 1] * 10 + sample_inputs[:, 2]
batch_b = sample_inputs[:, 6] * 100 + sample_inputs[:, 7] * 10 + sample_inputs[:, 8]
hybrid = np.bincount(carry_codes(batch_a, batch_b), minlength=8) / BATCH_SIZE
x = np.arange(8)
axes[1].bar(x - 0.2, natural, 0.4, label="natural")
axes[1].bar(x + 0.2, hybrid, 0.4, label="training batch")
axes[1].set_xticks(x, [f"{value:03b}" for value in x])
axes[1].set(xlabel="carry pattern", ylabel="fraction", title="Carry coverage")
axes[1].legend()
figure.tight_layout()
save_figure(figure, "addition_data_protocol")


## Dense transformer primitives


In [ ]:
def normal_parameter(rngs, shape, scale):
    values = jax.random.normal(rngs.params(), shape, dtype=jnp.float32) * scale
    return nnx.Param(values.astype(PARAM_DTYPE))

def matrix_multiply(x, weight):
    x16 = x.astype(COMPUTE_DTYPE)
    w16 = weight.astype(COMPUTE_DTYPE)
    return jax.lax.dot_general(
        x16, w16, (((x16.ndim - 1,), (0,)), ((), ())),
        preferred_element_type=jnp.float32,
    )

class Linear(nnx.Module):
    def __init__(self, input_width, output_width, rngs, scale=0.02):
        self.kernel = normal_parameter(rngs, (input_width, output_width), scale)

    def __call__(self, x):
        return matrix_multiply(x, self.kernel.value)

class LayerNorm(nnx.Module):
    def __init__(self, width):
        self.scale = nnx.Param(jnp.ones((width,), dtype=PARAM_DTYPE))
        self.bias = nnx.Param(jnp.zeros((width,), dtype=PARAM_DTYPE))

    def __call__(self, x):
        x = x.astype(jnp.float32)
        mean = jnp.mean(x, axis=-1, keepdims=True)
        variance = jnp.mean(jnp.square(x - mean), axis=-1, keepdims=True)
        return (x - mean) * jax.lax.rsqrt(variance + 1e-5) * self.scale.value + self.bias.value

def gelu(x):
    return 0.5 * x * (1.0 + jax.lax.erf(x / math.sqrt(2.0)))

CAUSAL_MASK = jnp.tril(jnp.ones((MODEL_INPUT_LENGTH, MODEL_INPUT_LENGTH), dtype=bool))

class CausalMHA(nnx.Module):
    def __init__(self, config, rngs):
        width = config["d_model"]
        residual_scale = 0.02 / math.sqrt(2 * config["n_layers"])
        self.width = width
        self.heads = config["n_heads"]
        self.head_dim = width // self.heads
        self.q_proj = Linear(width, width, rngs)
        self.k_proj = Linear(width, width, rngs)
        self.v_proj = Linear(width, width, rngs)
        self.out_proj = Linear(width, width, rngs, scale=residual_scale)

    def __call__(self, x):
        batch, length, _ = x.shape
        q = self.q_proj(x).reshape(batch, length, self.heads, self.head_dim)
        k = self.k_proj(x).reshape(batch, length, self.heads, self.head_dim)
        v = self.v_proj(x).reshape(batch, length, self.heads, self.head_dim)
        scores = jnp.einsum(
            "bthd,bshd->bhts", q.astype(jnp.float32), k.astype(jnp.float32),
            preferred_element_type=jnp.float32,
        ) / math.sqrt(self.head_dim)
        scores = jnp.where(
            CAUSAL_MASK[None, None, :length, :length],
            scores,
            jnp.finfo(jnp.float32).min,
        )
        probabilities = jax.nn.softmax(scores, axis=-1)
        attended = jnp.einsum(
            "bhts,bshd->bthd", probabilities, v.astype(jnp.float32),
            preferred_element_type=jnp.float32,
        )
        return self.out_proj(attended.reshape(batch, length, self.width))

class ModuleSequence(nnx.Module):
    def __init__(self, layers):
        self.length = len(layers)
        for index, layer in enumerate(layers):
            setattr(self, f"layer_{index}", layer)

    def __iter__(self):
        return (getattr(self, f"layer_{index}") for index in range(self.length))


## Four dense transformers and parameter verification


In [ ]:
class DenseFeedForward(nnx.Module):
    def __init__(self, config, rngs):
        residual_scale = 0.02 / math.sqrt(2 * config["n_layers"])
        self.up = Linear(config["d_model"], config["d_ff"], rngs)
        self.down = Linear(config["d_ff"], config["d_model"], rngs, scale=residual_scale)

    def __call__(self, x):
        return self.down(gelu(self.up(x)))

class DenseTransformerBlock(nnx.Module):
    def __init__(self, config, rngs):
        self.attention_norm = LayerNorm(config["d_model"])
        self.attention = CausalMHA(config, rngs)
        self.ffn_norm = LayerNorm(config["d_model"])
        self.ffn = DenseFeedForward(config, rngs)

    def __call__(self, x):
        x = x + self.attention(self.attention_norm(x))
        return x + self.ffn(self.ffn_norm(x))

class DenseAdditionTransformer(nnx.Module):
    def __init__(self, config, rngs):
        width = config["d_model"]
        self.token_embedding = normal_parameter(rngs, (VOCAB_SIZE, width), 0.02)
        self.position_embedding = normal_parameter(rngs, (MODEL_INPUT_LENGTH, width), 0.02)
        self.blocks = ModuleSequence([
            DenseTransformerBlock(config, rngs) for _ in range(config["n_layers"])
        ])
        self.final_norm = LayerNorm(width)

    def __call__(self, token_ids):
        x = self.token_embedding.value[token_ids]
        x = x + self.position_embedding.value[None, :, :]
        for block in self.blocks:
            def apply_block(value, current_block=block):
                return current_block(value)
            x = jax.checkpoint(apply_block)(x)
        x = self.final_norm(x)
        return jnp.einsum(
            "btd,vd->btv", x.astype(jnp.float32),
            self.token_embedding.value.astype(jnp.float32),
            preferred_element_type=jnp.float32,
        )

def dense_parameter_formula(config):
    layers, width, feed_forward = config["n_layers"], config["d_model"], config["d_ff"]
    return (
        layers * (4 * width * width + 2 * width * feed_forward + 4 * width)
        + VOCAB_SIZE * width + MODEL_INPUT_LENGTH * width + 2 * width
    )

parameter_rows = []
for model_id, config in DENSE_CONFIGS.items():
    model = DenseAdditionTransformer(config, nnx.Rngs(params=SEED))
    real_count = sum(int(leaf.size) for leaf in jax.tree.leaves(nnx.state(model, nnx.Param)))
    formula_count = dense_parameter_formula(config)
    assert real_count == formula_count == config["parameter_count"]
    parameter_rows.append({"model": model_id, "tree count": real_count,
                           "formula count": formula_count, "verified": True})
    del model
    gc.collect()

display(pd.DataFrame(parameter_rows).style.format({"tree count": "{:,}",
                                                   "formula count": "{:,}"}))


## Loss, generation, training, and validation


In [ ]:
ANSWER_MASK = jnp.arange(MODEL_INPUT_LENGTH) >= MODEL_INPUT_LENGTH - ANSWER_DIGITS

def answer_loss(logits, targets):
    log_probabilities = jax.nn.log_softmax(logits.astype(jnp.float32), axis=-1)
    token_losses = -jnp.take_along_axis(
        log_probabilities, targets[..., None], axis=-1
    ).squeeze(-1)
    denominator = targets.shape[0] * ANSWER_DIGITS
    loss = jnp.sum(jnp.where(ANSWER_MASK[None, :], token_losses, 0.0)) / denominator
    predictions = jnp.argmax(logits, axis=-1)
    accuracy = jnp.sum(
        jnp.where(ANSWER_MASK[None, :], predictions == targets, False)
    ) / denominator
    return loss, accuracy

def greedy_generate(model, prompts):
    buffer = jnp.zeros((prompts.shape[0], MODEL_INPUT_LENGTH), dtype=jnp.int32)
    buffer = buffer.at[:, :PROMPT_LENGTH].set(prompts)

    def generate_digit(current_buffer, offset):
        logits = model(current_buffer)
        next_token = jnp.argmax(logits[:, PROMPT_LENGTH + offset - 1, :], axis=-1)
        current_buffer = jax.lax.cond(
            offset < ANSWER_DIGITS - 1,
            lambda value: value.at[:, PROMPT_LENGTH + offset].set(next_token),
            lambda value: value,
            current_buffer,
        )
        return current_buffer, next_token.astype(jnp.int32)

    _, generated = jax.lax.scan(generate_digit, buffer, jnp.arange(ANSWER_DIGITS))
    generated = jnp.swapaxes(generated, 0, 1)
    return generated, jnp.all(generated < 10, axis=-1)

def make_learning_rate(horizon):
    warmup = max(1, horizon // 10)
    schedule = optax.warmup_cosine_decay_schedule(
        init_value=0.0,
        peak_value=PEAK_LR,
        warmup_steps=warmup,
        decay_steps=horizon - 1,
        end_value=FINAL_LR,
    )
    assert float(schedule(0)) == 0.0
    np.testing.assert_allclose(float(schedule(horizon - 1)), FINAL_LR, rtol=1e-5)
    return schedule

def path_parts(path):
    return tuple(str(getattr(entry, "key", getattr(entry, "idx", entry))) for entry in path)

def all_finite(tree):
    return jnp.all(jnp.stack([jnp.all(jnp.isfinite(leaf)) for leaf in jax.tree.leaves(tree)]))


In [ ]:
DENSE_ROOT = DRIVE_ROOT / "scaling" / "notebook_02_dense"
DENSE_RUNS_ROOT = DENSE_ROOT / "runs"
DENSE_RESULTS_PATH = DENSE_ROOT / "dense_results.json"
DENSE_RUNS_ROOT.mkdir(parents=True, exist_ok=True)

def build_dense_training(config, horizon):
    model = DenseAdditionTransformer(config, nnx.Rngs(params=SEED))
    graphdef, params = nnx.split(model, nnx.Param)
    decay_mask = jax.tree_util.tree_map_with_path(
        lambda path, leaf: leaf.ndim == 2 and (
            "attention" in path_parts(path) or "ffn" in path_parts(path)
        ),
        params,
    )
    learning_rate = make_learning_rate(horizon)
    optimizer = optax.chain(
        optax.clip_by_global_norm(GRAD_CLIP),
        optax.scale_by_adam(b1=0.9, b2=0.99, eps=1e-8),
        optax.masked(optax.add_decayed_weights(WEIGHT_DECAY), decay_mask),
        optax.scale_by_learning_rate(learning_rate),
    )
    optimizer_state = optimizer.init(params)

    @jax.jit
    def train_step(params, optimizer_state, inputs, targets):
        def objective(candidate_params):
            current_model = nnx.merge(graphdef, candidate_params)
            return answer_loss(current_model(inputs), targets)
        (loss, accuracy), gradients = jax.value_and_grad(objective, has_aux=True)(params)
        gradient_norm = optax.global_norm(gradients)
        updates, new_optimizer_state = optimizer.update(gradients, optimizer_state, params)
        new_params = optax.apply_updates(params, updates)
        finite = jnp.isfinite(loss) & all_finite(gradients) & all_finite(new_params)
        safe_params = jax.tree.map(lambda new, old: jnp.where(finite, new, old),
                                   new_params, params)
        safe_state = jax.tree.map(lambda new, old: jnp.where(finite, new, old),
                                  new_optimizer_state, optimizer_state)
        return safe_params, safe_state, {
            "loss": loss, "answer_token_accuracy": accuracy,
            "gradient_norm": gradient_norm, "finite": finite,
        }

    @jax.jit
    def evaluate_batch(params, inputs, targets):
        return answer_loss(nnx.merge(graphdef, params)(inputs), targets)

    @jax.jit
    def generate_batch(params, prompts):
        return greedy_generate(nnx.merge(graphdef, params), prompts)

    return params, optimizer_state, learning_rate, train_step, evaluate_batch, generate_batch

def evaluate_validation(params, evaluate_batch, generate_batch):
    started = time.perf_counter()
    loss_total = token_total = 0.0
    exact_total = 0
    for start in range(0, len(validation_ids), EVAL_BATCH_SIZE):
        ids = validation_ids[start:start + EVAL_BATCH_SIZE]
        sequences = make_sequences(ids).astype(np.int32)
        inputs = jnp.asarray(sequences[:, :-1])
        targets = jnp.asarray(sequences[:, 1:])
        loss, token_accuracy = evaluate_batch(params, inputs, targets)
        generated, valid = generate_batch(params, inputs[:, :PROMPT_LENGTH])
        generated, valid = np.asarray(generated), np.asarray(valid)
        exact = valid & np.all(generated == sequences[:, -ANSWER_DIGITS:], axis=1)
        loss_total += float(loss) * len(ids)
        token_total += float(token_accuracy) * len(ids)
        exact_total += int(exact.sum())
    return {
        "validation_loss": loss_total / len(validation_ids),
        "validation_token_accuracy": token_total / len(validation_ids),
        "validation_exact_match": exact_total / len(validation_ids),
        "validation_seconds": time.perf_counter() - started,
    }

def run_dense_experiment(model_id, config, horizon):
    run_id = f"{model_id}_h{horizon:04d}"
    run_config = {"run_id": run_id, "model_id": model_id, "horizon": horizon,
                  "seed": SEED, "batch_size": BATCH_SIZE, "model": config}
    run_dir = DENSE_RUNS_ROOT / run_id
    latest_path = run_dir / "latest_checkpoint.pkl"
    best_path = run_dir / "best_checkpoint.pkl"
    history_path = run_dir / "history.json"
    result_path = run_dir / "result.json"
    run_dir.mkdir(parents=True, exist_ok=True)

    if result_path.exists():
        completed = json.loads(result_path.read_text())
        if completed.get("status") == "complete" and completed.get("run_config") == run_config:
            print(f"skip {run_id}: complete")
            return completed
        raise ValueError(f"existing result does not match this run: {run_id}")

    params, optimizer_state, learning_rate, train_step, evaluate_batch, generate_batch = (
        build_dense_training(config, horizon)
    )
    batcher = HybridBatcher(train_sequences, train_strata, BATCH_SIZE, SEED)
    history = []
    start_step = 0
    optimizer_seconds = 0.0
    compilation_seconds = 0.0

    if latest_path.exists():
        checkpoint = pickle.loads(latest_path.read_bytes())
        if checkpoint["run_config"] != run_config:
            raise ValueError(f"checkpoint configuration mismatch: {run_id}")
        params = jax.tree.map(jnp.asarray, checkpoint["params"])
        optimizer_state = jax.tree.map(jnp.asarray, checkpoint["optimizer_state"])
        batcher.rng.bit_generator.state = checkpoint["batcher_rng_state"]
        history = checkpoint["history"]
        start_step = checkpoint["step"]
        optimizer_seconds = checkpoint["optimizer_seconds"]
        compilation_seconds = checkpoint["compilation_seconds"]
        print(f"resume {run_id} after step {start_step}")
    else:
        print(f"start {run_id}")

    checkpoint_every = min(50, horizon)
    log_every = min(25, horizon)
    for step in range(start_step + 1, horizon + 1):
        inputs, targets = batcher.sample()
        before = time.perf_counter()
        params, optimizer_state, metrics = train_step(
            params, optimizer_state, jnp.asarray(inputs), jnp.asarray(targets)
        )
        jax.block_until_ready(metrics["loss"])
        elapsed = time.perf_counter() - before
        optimizer_seconds += elapsed
        if step == start_step + 1:
            compilation_seconds += elapsed

        if step == start_step + 1 or step % log_every == 0 or step == horizon:
            host = jax.device_get(metrics)
            assert bool(host["finite"]), f"non-finite values at {run_id} step {step}"
            record = {
                "step": step, "loss": float(host["loss"]),
                "answer_token_accuracy": float(host["answer_token_accuracy"]),
                "gradient_norm": float(host["gradient_norm"]),
                "learning_rate": float(learning_rate(step - 1)),
                "step_seconds": elapsed,
            }
            history.append(record)
            print(
                f"{run_id} | step {step:4d}/{horizon} | loss {record['loss']:.4f} | "
                f"token acc {100 * record['answer_token_accuracy']:6.2f}% | "
                f"{elapsed:.3f}s"
            )

        if step % checkpoint_every == 0 or step == horizon:
            payload = {
                "run_config": run_config, "step": step,
                "params": jax.device_get(params),
                "optimizer_state": jax.device_get(optimizer_state),
                "batcher_rng_state": batcher.rng.bit_generator.state,
                "history": history, "optimizer_seconds": optimizer_seconds,
                "compilation_seconds": compilation_seconds,
            }
            atomic_write_bytes(latest_path, pickle.dumps(payload, protocol=pickle.HIGHEST_PROTOCOL))
            atomic_write_json(history_path, history)
            print(f"saved checkpoint: {run_id} step {step}")

    validation = evaluate_validation(params, evaluate_batch, generate_batch)
    parameter_count = config["parameter_count"]
    examples_seen = horizon * BATCH_SIZE
    result = {
        "status": "complete", "run_config": run_config,
        "run_id": run_id, "architecture": "dense", "model_id": model_id,
        "horizon": horizon, "parameter_count": parameter_count,
        "examples_seen": examples_seen,
        "input_tokens": examples_seen * MODEL_INPUT_LENGTH,
        "answer_tokens": examples_seen * ANSWER_DIGITS,
        "estimated_training_flops": 6 * parameter_count * examples_seen * MODEL_INPUT_LENGTH,
        "final_training_loss": history[-1]["loss"],
        "final_training_token_accuracy": history[-1]["answer_token_accuracy"],
        "optimizer_seconds": optimizer_seconds,
        "compilation_seconds": compilation_seconds,
        **validation,
    }
    atomic_write_bytes(best_path, pickle.dumps({
        "run_config": run_config, "params": jax.device_get(params), "validation": validation
    }, protocol=pickle.HIGHEST_PROTOCOL))
    atomic_write_json(result_path, result)
    print(
        f"complete {run_id} | validation loss {validation['validation_loss']:.4f} | "
        f"exact {100 * validation['validation_exact_match']:.2f}%"
    )
    return result


## Run the 16 independent experiments


In [ ]:
dense_results = []
for model_id, config in DENSE_CONFIGS.items():
    for horizon in HORIZONS:
        dense_results.append(run_dense_experiment(model_id, config, horizon))
        atomic_write_json(DENSE_RESULTS_PATH, {
            "status": "complete" if len(dense_results) == 16 else "in_progress",
            "results": dense_results,
        })
        gc.collect()
        jax.clear_caches()

assert len(dense_results) == 16
assert all(result["status"] == "complete" for result in dense_results)
atomic_write_json(DENSE_RESULTS_PATH, {"status": "complete", "results": dense_results})
print("Dense study complete:", DENSE_RESULTS_PATH)


## Result tables and figures


In [ ]:
dense_frame = pd.DataFrame(dense_results).sort_values(["parameter_count", "horizon"])

quality_table = dense_frame[[
    "model_id", "horizon", "validation_loss", "validation_token_accuracy",
    "validation_exact_match", "final_training_loss",
]].copy()
quality_table["validation_token_accuracy"] *= 100
quality_table["validation_exact_match"] *= 100

exposure_compute_table = dense_frame[[
    "model_id", "horizon", "examples_seen", "input_tokens", "answer_tokens",
    "estimated_training_flops", "optimizer_seconds", "validation_seconds",
]].copy()
exposure_compute_table["examples_per_second"] = (
    exposure_compute_table["examples_seen"] / exposure_compute_table["optimizer_seconds"]
)

display(quality_table.style.format({
    "validation_loss": "{:.5f}", "validation_token_accuracy": "{:.2f}%",
    "validation_exact_match": "{:.2f}%", "final_training_loss": "{:.5f}",
}))
display(exposure_compute_table.style.format({
    "examples_seen": "{:,}", "input_tokens": "{:,}", "answer_tokens": "{:,}",
    "estimated_training_flops": "{:.3e}", "optimizer_seconds": "{:.1f}",
    "validation_seconds": "{:.1f}", "examples_per_second": "{:,.0f}",
}))

figure, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for model_id, group in dense_frame.groupby("model_id", sort=False):
    axes[0].plot(group["input_tokens"], group["validation_loss"], marker="o", label=model_id)
    axes[1].plot(group["input_tokens"], 100 * group["validation_exact_match"], marker="o")
    axes[2].plot(group["horizon"], group["optimizer_seconds"], marker="o")
axes[0].set(xscale="log", yscale="log", xlabel="input-token exposure",
            ylabel="validation loss", title="Dense validation loss")
axes[0].legend(fontsize=8)
axes[1].set(xscale="log", xlabel="input-token exposure", ylabel="exact match (%)",
            title="Dense exact match")
axes[2].set(xlabel="independent horizon", ylabel="optimizer seconds", title="Measured time")
figure.tight_layout()
save_figure(figure, "dense_scaling_results")


## Empirical frontier and scaling-law fit


In [ ]:
def empirical_frontier(frame):
    ordered = frame.sort_values(["estimated_training_flops", "validation_loss"])
    keep, best_loss = [], math.inf
    for index, row in ordered.iterrows():
        if row["validation_loss"] < best_loss:
            keep.append(index)
            best_loss = row["validation_loss"]
    return ordered.loc[keep]

frontier = empirical_frontier(dense_frame)
fit_frame = dense_frame[dense_frame["validation_loss"] > 0].copy()
design = np.column_stack([
    np.ones(len(fit_frame)),
    -np.log(fit_frame["parameter_count"].to_numpy() / 1e6),
    -np.log(fit_frame["input_tokens"].to_numpy() / 1e6),
])
target = np.log(fit_frame["validation_loss"].to_numpy())
coefficient, *_ = np.linalg.lstsq(design, target, rcond=None)
prediction = np.exp(design @ coefficient)
residual = target - np.log(prediction)
log_r_squared = 1 - np.sum(residual ** 2) / np.sum((target - target.mean()) ** 2)
scaling_fit = {
    "amplitude": float(np.exp(coefficient[0])),
    "parameter_exponent": float(coefficient[1]),
    "token_exponent": float(coefficient[2]),
    "log_space_r_squared": float(log_r_squared),
}
atomic_write_json(DENSE_ROOT / "scaling_fit.json", scaling_fit)

display(frontier[["model_id", "horizon", "estimated_training_flops",
                  "validation_loss", "validation_exact_match"]].style.format({
    "estimated_training_flops": "{:.3e}", "validation_loss": "{:.5f}",
    "validation_exact_match": "{:.2%}",
}))
display(pd.DataFrame([scaling_fit]).style.format("{:.4f}"))

figure, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(frontier["estimated_training_flops"], frontier["validation_loss"], marker="o")
axes[0].set(xscale="log", yscale="log", xlabel="estimated training FLOPs",
            ylabel="validation loss", title="Empirical compute frontier")
axes[1].scatter(fit_frame["validation_loss"], prediction,
                c=np.log10(fit_frame["parameter_count"]), cmap="viridis")
limits = [min(fit_frame["validation_loss"].min(), prediction.min()),
          max(fit_frame["validation_loss"].max(), prediction.max())]
axes[1].plot(limits, limits, linestyle="--", color="black", linewidth=1)
axes[1].set(xscale="log", yscale="log", xlabel="measured loss", ylabel="fitted loss",
            title="Power-law fit")
figure.tight_layout()
save_figure(figure, "dense_frontier_and_fit")


## Conclusions from completed results


In [ ]:
assert len(dense_frame) == 16 and dense_frame["validation_loss"].notna().all()
best = dense_frame.loc[dense_frame["validation_loss"].idxmin()]
threshold = dense_frame[dense_frame["validation_exact_match"] >= 0.95]
print("Conclusions from completed dense runs")
print(
    f"Lowest validation loss: {best['model_id']} at {int(best['horizon'])} steps "
    f"({best['validation_loss']:.5f})."
)
if len(threshold):
    first = threshold.sort_values(["input_tokens", "parameter_count"]).iloc[0]
    print(
        f"First observed ≥95% exact match: {first['model_id']} at "
        f"{int(first['horizon'])} steps ({int(first['input_tokens']):,} input tokens)."
    )
else:
    print("No completed coordinate reached 95% validation exact match.")
print(
    "Fitted exponents on these 16 observations: "
    f"parameters={scaling_fit['parameter_exponent']:.3f}, "
    f"tokens={scaling_fit['token_exponent']:.3f}, "
    f"log-space R²={scaling_fit['log_space_r_squared']:.3f}."
)
